# Ajuste de hiper-parámetros

Cuando trabajamos con modelos de *machine learning* a menudo nos encontramos con que estos disponen de una serie de hiper-parámetros que es necesario ajustar con el fin de lograr que realicen las predicciones más certeras posibles sobre nuestro conjunto de datos. A este paso se le conoce como **hyper-parameters tunning**.

Veamos cómo podemos ajustar los hiper-parámetros de un `RandomForestClassifier`para el conjunto de datos de las caras de Olivetti.

Cargamos las caras:

In [ ]:
from sklearn.datasets import fetch_olivetti_faces

olivetti = fetch_olivetti_faces()

In [ ]:
X = olivetti.data
y = olivetti.target

El conjunto de datos de las caras de Olivetti busca identificar a 40 personas mediante imágenes de sus rostros.

In [ ]:
print(olivetti.DESCR)

Algunas de las caras:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_images = 6
image_shape = (64, 64)

random_indices = np.random.choice(X.shape[0], size=n_images, replace=False)
faces = X[random_indices, :]

fig, axs = plt.subplots(nrows=1, ncols=n_images, figsize=(3*n_images, 3))

for i, comp in enumerate(faces):
    axs[i].imshow(comp.reshape(image_shape), cmap=plt.cm.gray, interpolation='nearest', vmin=0, vmax=1)
    axs[i].set_xticks(())
    axs[i].set_yticks(())

Prepramos el dataset y lo partimos en `train` y `test`:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## Optimización de un hiper-parámetro

Si quieremos optimizar un único híper-parametro, lo ideal es analizar cómo se comporta una medida de calidad según evoluciona dicho hiper-parámetro.

Por ejemplo, veamos como varía el *accuracy* al variar el hiper-parámetro `n_estimators`:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier

n_estimators = np.arange(1,50,3)
train_scores = np.array([])
test_scores = np.array([])

for n in n_estimators:
  clf = RandomForestClassifier(n_estimators=n, verbose=1, n_jobs=-1, random_state=42).fit(X_train, y_train)
  train_scores = np.append(train_scores, clf.score(X_train, y_train))
  test_scores = np.append(test_scores, clf.score(X_test, y_test))

plt.figure()

plt.plot(n_estimators, train_scores, label="train")
plt.plot(n_estimators, test_scores, label="test")

plt.xlabel('n_estimators')
plt.ylabel('accuracy')

plt.legend()

## Optimización de dos hiper-parámetros

Si queremos comprobar cómo evoluciona el error cuando para las variaciones producidas por dos hiper-parámetros, debemos pintar un mapa de calor en el que se muestre el resultado de evaluar el modelo para las combinaciones de calores de los dos hiper-parámetros.

Veamos como evoluciona el *accuracy* cuando variamos `n_estimators` y `max_depth`:

In [ ]:
import matplotlib.pyplot as plt

n_estimators = np.arange(5,51,5)
max_depths = np.arange(1,30,3)

train_scores = np.array([])
test_scores = np.array([])

for n in n_estimators:
  for d in max_depths:
    clf = RandomForestClassifier(n_estimators=n, max_depth=d, verbose=1, n_jobs=-1, random_state=42).fit(X_train, y_train)
    train_scores = np.append(train_scores, clf.score(X_train, y_train))
    test_scores = np.append(test_scores, clf.score(X_test, y_test))

fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10,5))

# train error 

im0 = axs[0].imshow(train_scores.reshape((len(n_estimators), len(max_depths))), vmin=0, vmax=1, cmap='Blues')
plt.colorbar(im0, ax=axs[0])

axs[0].set_xlabel('max_depths')
axs[0].set_xticks(np.arange(len(max_depths)))
axs[0].set_xticklabels(max_depths)

axs[0].set_ylabel('n_estimators')
axs[0].set_yticks(np.arange(len(n_estimators)))
axs[0].set_yticklabels(n_estimators)

axs[0].set_title('Train')

# test error

im1 = axs[1].imshow(test_scores.reshape((len(n_estimators), len(max_depths))), vmin=0, vmax=1, cmap='Blues')
plt.colorbar(im1, ax=axs[1])

axs[1].set_xlabel('max_depths')
axs[1].set_xticks(np.arange(len(max_depths)))
axs[1].set_xticklabels(max_depths)

axs[1].set_ylabel('n_estimators')
axs[1].set_yticks(np.arange(len(n_estimators)))
axs[1].set_yticklabels(n_estimators)

axs[1].set_title('Test')

## Optimización de múltiples hiper-parámetros

Para analizar qué sucede con más de dos hiper-parámetros, no podemos pintar ningún tipo de gráfico al tener más de tres dimensiona a representar (dos para los hiper-parámetros y una para el error). Por ello, en lugar de representar gráficamente el error lo hacemos de forma tabulada.

[`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) es una excelente herramienta que prueba de forma exahstiva todas las combinaciones de hiper-parámetros para unos rangos pre-seleccionados.

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {
  'n_estimators': np.arange(5,51,10),
  'max_depth': np.arange(1,30,5),
  'criterion': ('gini', 'entropy')
}

rf = RandomForestClassifier(random_state=42)
gs = GridSearchCV(rf, parameters, verbose=1, n_jobs=-1, cv=3)
gs.fit(X, y)

In [ ]:
import pandas as pd
pd.DataFrame(gs.cv_results_).sort_values('rank_test_score')

El problema de *Grid Search* es que es profundamente explorativo, por lo que los tiempos de ejecución suelen dispararse. Como alternativa disponemos de [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) que selecciona aleatoriamente combinaciones de hiper-parámetros dentro del rango elegido.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

parameters = {
  'n_estimators': np.arange(5,51,5),
  'max_depth': np.arange(1,30,3),
  'criterion': ('gini', 'entropy')
}

rf =  RandomForestClassifier(random_state=42)
rs = RandomizedSearchCV(rf, parameters, n_iter=10, verbose=1, n_jobs=-1, cv=3)
rs.fit(X_train, y_train)

In [ ]:
import pandas as pd
pd.DataFrame(rs.cv_results_).sort_values('rank_test_score')

# Optimización de hiper-parámetros con Nevergrad

[Nevergrad](https://facebookresearch.github.io/nevergrad/index.html) es una librería de optimización numérica de código abierto desarrollada por **Facebook**. Aunque puede utilizarse para resolver cualquier problema de optimización, está diseñada para realizar una optimización de hiper-parámetros cuando estamos ajustando un modelo.

A diferencia de `GridSearch`, **Nevergrad** no realiza una búsqueda exhaustiva sobre todas las combinaciones posibles de valores, sino que utiliza metaheurísticas tales como los algoritmos evolutivos y sus variantes.

In [ ]:
!pip install nevergrad

Un ejemplo rápido del funcionamiento de la librería sería optimizar una función bidimensional:

In [ ]:
import nevergrad as ng

def square(x):
    return sum((x - 0.5) ** 2)

# optimization on x as an array of shape (2,)
optimizer = ng.optimizers.NGOpt(parametrization=2, budget=100)
recommendation = optimizer.minimize(square)  # best value
print(recommendation.value)

Vamos a optimizar los hiper-parámetros del modelo anterior, pero usando esta vez la librería **Nevergrad**. Para que funcione, necesitamos una función de evaluación para medir la calidad de un conjunto determinado de hiper-parámetros. Para ello encapsulamos la construcción del modelo junto con la evaluación del mismo (usaremos *F1-score*):

In [ ]:
from sklearn.metrics import f1_score

def create_and_eval_rf(n_estimators, max_depth, criterion, X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test):
    model = RandomForestClassifier(n_estimators=n_estimators,
                                   max_depth=max_depth,
                                   criterion=criterion,
                                   random_state=42)
    model.fit(X_train, y_train)
    score = f1_score(y_test, model.predict(X_test), average='weighted')
    print(f"F1-score: {score}")
    return -score


Como podemos observar, los hiper-parámetros del modelo son los siguientes:

- `n_estimators`: número de árboles de decisión a ajustar.
- `max_depth`: profundidad máxima de cada árbol.
- `criterion`: criterio de selección del punto de corte.

**Nevergrad** denomina **intrumentalización** a la definición de los hiper-parámetros. Observa que los nombres de los parámetros de la instrumentalización coinciden con los parámetros de la función de evaluación. También es necesario elegir uno de los [optimizadores](https://facebookresearch.github.io/nevergrad/optimizers_ref.html#optimizers) que vienen incluidos en **Nevergrad**.

Por último, bastará con ejecutar la meta-optimización consistente en minimizar la función de evaluación que hemos definido anteriormente.

In [ ]:
import nevergrad as ng

n_estimators = ng.p.TransitionChoice(range(10, 100, 10))
max_depth = ng.p.TransitionChoice([3, 4, 5])
criterion = ng.p.Choice(['gini', 'entropy'])

params = ng.p.Instrumentation(n_estimators, max_depth, criterion)
optimizer = ng.optimizers.TwoPointsDE(parametrization=params, budget=100)
best = optimizer.minimize(create_and_eval_rf, batch_mode=False)

In [ ]:
best.value

---

Creado por **Fernando Ortega** (fernando.ortega@upm.es) y **Raúl Lara-Cabrera** (raul.lara@upm.es)

<img src="https://licensebuttons.net/l/by-nc-sa/3.0/88x31.png">